In [ ]:
import pandas as pd
import joblib
import shap
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------
# LOAD ARTIFACTS (LOAD ONCE)
# -------------------------------
model = joblib.load("model.pkl")
tfidf = joblib.load("tfidf.pkl")
columns = joblib.load("columns.pkl")

explainer = shap.TreeExplainer(model)

# -------------------------------
# BUILD FEATURE ROW
# -------------------------------
def build_feature_row(startup_data, investor_data):

    df = pd.DataFrame([{
        "startup_industry": startup_data["industry"],
        "startup_stage": startup_data["stage"],
        "funding_required_lakhs": startup_data["funding_required"],

        "investor_pref_industry": investor_data["preferred_industry"],
        "investor_pref_stage": investor_data["preferred_stage"],
        "investor_ticket_min_lakhs": investor_data["ticket_min"],
        "investor_ticket_max_lakhs": investor_data["ticket_max"],

        "startup_desc": startup_data["description"],
        "investor_desc": investor_data["description"]
    }])

    # Idea similarity
    s_vec = tfidf.transform(df["startup_desc"])
    i_vec = tfidf.transform(df["investor_desc"])
    similarity = cosine_similarity(s_vec, i_vec)[0][0]

    df["idea_similarity"] = similarity

    # Funding fit
    df["funding_fit"] = (
        (df["funding_required_lakhs"] >= df["investor_ticket_min_lakhs"]) &
        (df["funding_required_lakhs"] <= df["investor_ticket_max_lakhs"])
    ).astype(int)

    df = df.drop(["startup_desc", "investor_desc"], axis=1)

    df_encoded = pd.get_dummies(df, drop_first=True)

    df_aligned = df_encoded.reindex(columns=columns, fill_value=0)

    return df_aligned

# -------------------------------
# EXPLANATION ENGINE (FIXED)
# -------------------------------
def explain_match(startup_data, investor_data):

    X_input = build_feature_row(startup_data, investor_data)

    prediction = float(model.predict(X_input)[0])
    prediction = float(max(0, min(100, prediction)))

    shap_vals = explainer.shap_values(X_input)[0]
    feature_contrib = dict(zip(X_input.columns, shap_vals))

    # Explicit mapping ⭐⭐⭐⭐⭐
    industry_contrib = sum(v for k, v in feature_contrib.items() if "industry" in k.lower())
    stage_contrib = sum(v for k, v in feature_contrib.items() if "stage" in k.lower())
    funding_contrib = feature_contrib.get("funding_fit", 0)
    similarity_contrib = feature_contrib.get("idea_similarity", 0)

    contributions_raw = {
        "Industry Fit": abs(industry_contrib),
        "Stage Fit": abs(stage_contrib),
        "Funding Fit": abs(funding_contrib),
        "Idea Similarity": abs(similarity_contrib)
    }

    # Normalize safely ⭐⭐⭐⭐⭐
    total = sum(contributions_raw.values()) + 1e-6

    contributions_pct = {
        k: round(100 * v / total, 1)
        for k, v in contributions_raw.items()
    }

    return {
        "match_score": round(prediction, 2),
        "contributions": contributions_pct
    }